# Basic Usage

This notebook shows basic usage of cuCIM with a whole-slide pathology image.

## Prerequisites

In [ ]:
# !conda install -c conda-forge pillow

## Read image

In [ ]:
from cucim import CuImage

img = CuImage("/home/azureuser/nvidia/cucim_notebooks/input/image.tif")
print(f"Image loaded: {img.path}")
print(f"Dimensions (XY): {img.size('XY')}")

### See metadata

In [ ]:
import json

metadata = img.metadata
print(json.dumps(metadata, indent=2)[:2000])

### Read region

Let's read the whole slide at the lowest resolution and save as a thumbnail.

In [ ]:
from PIL import Image

resolutions = img.resolutions
level_dimensions = resolutions["level_dimensions"]
level_count = resolutions["level_count"]

print(f"Number of levels: {level_count}")
print(f"Level dimensions: {level_dimensions}")

# Read the whole slide at lowest resolution
region = img.read_region(
    location=[0, 0], size=level_dimensions[level_count - 1], level=level_count - 1
)

# Save and display
region.save("/home/azureuser/nvidia/cucim_notebooks/input/thumbnail.ppm")
Image.open("/home/azureuser/nvidia/cucim_notebooks/input/thumbnail.ppm")

Now let's read a 512x512 region at full resolution (level 0).

In [ ]:
from PIL import Image

region = img.read_region([10000, 20000], [512, 512], 0)
region.save("/home/azureuser/nvidia/cucim_notebooks/input/test.ppm")
Image.open("/home/azureuser/nvidia/cucim_notebooks/input/test.ppm")

### `__array_interface__` support

A NumPy array has the `__array_interface__` property. Here is a simple example:

In [ ]:
import numpy as np

np_arr = np.array([1, 2, 3])
print(np_arr.__array_interface__)

As you can see from the above result, a NumPy array has `__array_interface__` property which describes the memory layout.

**cuCIM also supports `__array_interface__`**, so you can directly convert a cuCIM region to a NumPy array and display it with PIL:

In [ ]:
import numpy as np
from PIL import Image

region = img.read_region([10000, 20000], [512, 512], 0)
np_img_arr = np.asarray(region)

print(f"Array shape: {np_img_arr.shape}, dtype: {np_img_arr.dtype}")
arr_iface = np_img_arr.__array_interface__
print(f"__array_interface__ shape: {arr_iface['shape']}")

Image.fromarray(np_img_arr)

### `__cuda_array_interface__` support

A CuPy array has `__cuda_array_interface__` property. Here is a simple example:

In [ ]:
import cupy as cp

cp_arr = cp.array([1, 2, 3])
print(cp_arr.__cuda_array_interface__)

A Python object that has `__cuda_array_interface__` property is considered as a CUDA array-like object and can be converted to CuPy array through `cp.asarray(obj)` method.

**`__cuda_array_interface__` is also supported in CuImage** if you specify `device='cuda'` in `read_region()`. The following code loads image data directly to GPU memory and visualizes it in the Jupyter notebook:

In [ ]:
import cupy as cp
from cucim import CuImage
from PIL import Image

img = CuImage("/home/azureuser/nvidia/cucim_notebooks/input/image.tif")
resolutions = img.resolutions
level_dimensions = resolutions["level_dimensions"]
level_count = resolutions["level_count"]

# Read region directly to GPU memory
region_gpu = img.read_region(
    [0, 0], level_dimensions[level_count - 1], level_count - 1, device="cuda"
)

print(f"Device: {region_gpu.device}")
cuda_iface = region_gpu.__cuda_array_interface__
print(f"__cuda_array_interface__ shape: {cuda_iface['shape']}")

# Convert to CuPy array, then to CPU for display
cupy_arr = cp.asarray(region_gpu)
print(f"CuPy array shape: {cupy_arr.shape}")
Image.fromarray(cupy_arr.get())

### Associated images

Some image formats such as Philips TIFF and Aperio SVS have associated images (Macro or Label images) in addition to the multi-resolution images.

Let's check what associated images are available:

In [ ]:
import numpy as np
from PIL import Image

print(f"Associated images: {img.associated_images}")

for name in img.associated_images:
    assoc_img = img.associated_image(name)
    np_arr = np.asarray(assoc_img)
    print(f"  '{name}': shape={np_arr.shape}")

# Display the macro image if available
if 'macro' in img.associated_images:
    macro_image = img.associated_image('macro')
    np_img_arr = np.asarray(macro_image)
    print(f"\nDisplaying macro image: {np_img_arr.shape}")
    display(Image.fromarray(np_img_arr))

# Check for label image
if 'label' in img.associated_images:
    print("Label image exists")
else:
    print("\nThere is no associated image named 'label'!")